# Phase 4 & 5: Remaining Useful Life (RUL) Prediction & Evaluation

This notebook implements and compares **12 RUL prediction models** across 4 model families on the NASA/IMS bearing dataset:

1. **Baselines**: Naive Health-Indicator (`HIBaseline`), `RidgeRUL`
2. **Classical ML**: `RandomForestRUL`, `XGBoostRUL`, `LightGBMRUL`, `SVRRUL`
3. **Deep Learning (PyTorch)**: `MLPRUL`, `CNN1DRUL`, `LSTMRUL`, `GRURUL`, `CNNLSTMRUL`
4. **Degradation-Model Based**: `ExpDegradationRUL`

### Experimental Design
* **Piecewise-linear RUL labels**: Capped at `RUL_CAP = 50 h` (median onset-to-end duration across the 4 runs) while healthy, then declining linearly to 0.
* **Validation**: **Leave-One-Run-Out (LORO) 4-fold cross-validation** (`set1_b3`, `set1_b4`, `set2_b1`, `set3_b3`). Scalers are fit strictly on the 3 training runs; sliding windows (`W=20`) are built strictly within each run.
* **Metrics**: RMSE, MAE, **PHM asymmetric score** (mean per snapshot, penalizing late predictions more than early ones), and late-life accuracy (`RUL <= 25 h`).
* **Ablations**: Causal prediction smoothing (`win=15`) and calendar vs. operating-hours labels.


In [ ]:
from pathlib import Path
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust project root discovery
root = Path.cwd().resolve()
while root != root.parent and not (root / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    for c in [Path("..").resolve(), Path(".").resolve(), Path(r"c:/Users/HP/Desktop/bearing-rul-project")]:
        if (c / "src").is_dir():
            root = c
            break
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.features import add_spectral_features
from src.health import prepare
from src.labels import RUNS, run_table
from src.evaluate import evaluate_rul_loro, rmse, mae, phm_score, smooth_predictions
from src.models_rul import (
    HIBaseline, RidgeRUL,
    RandomForestRUL, XGBoostRUL, LightGBMRUL, SVRRUL,
    ExpDegradationRUL,
    MLPRUL, CNN1DRUL, LSTMRUL, GRURUL, CNNLSTMRUL,
    _HAS_XGB, _HAS_LGBM, _HAS_TORCH,
)

print(f"Backends -> XGBoost: {_HAS_XGB} | LightGBM: {_HAS_LGBM} | PyTorch: {_HAS_TORCH}")

data_dir = root / "data"
results_dir = root / "results"
results_dir.mkdir(exist_ok=True, parents=True)

df = pd.read_parquet(data_dir / "processed_features.parquet")
spec = np.load(data_dir / "env_spectra.npy")
df = add_spectral_features(df, spec)

FEATURES = (["rms", "peak", "kurt", "skew", "crest", "shape", "impulse",
             "spec_entropy", "spec_centroid"]
            + [f"band{i}_frac" for i in range(7)])

z_runs = prepare(df, FEATURES)
tables = {f"{s}_{b}": run_table(df, s, b) for s, b in RUNS}

RUL_CAP = 50.0  # hours (see results/data_notes.md)
runs_cal = {}
runs_op  = {}
for name, Z in z_runs.items():
    t = tables[name].set_index("idx").loc[Z.index]
    X = Z[FEATURES].to_numpy(dtype=np.float32)
    runs_cal[name] = (X, np.clip(t["rul_hours"].to_numpy(dtype=np.float32), 0.0, RUL_CAP))
    runs_op[name]  = (X, np.clip(t["rul_op_hours"].to_numpy(dtype=np.float32), 0.0, RUL_CAP))
    print(f"  {name}: {X.shape}, cal_rul <= cap fraction = {(runs_cal[name][1] < RUL_CAP).mean():.2%}")


---
## 1. Leave-One-Run-Out (LORO) Cross-Validation — All 12 Models


In [ ]:
RUL_MAKERS = {
    "hi_baseline":     HIBaseline,
    "ridge":           RidgeRUL,
    "random_forest":   RandomForestRUL,
    "xgboost":         XGBoostRUL,
    "lightgbm":        LightGBMRUL,
    "svr":             SVRRUL,
    "exp_degradation": ExpDegradationRUL,
    "mlp":             MLPRUL,
    "cnn1d":           CNN1DRUL,
    "lstm":            LSTMRUL,
    "gru":             GRURUL,
    "cnn_lstm":        CNNLSTMRUL,
}

all_rows = []
all_preds = {}
for mname, mk in RUL_MAKERS.items():
    print(f"Running LORO CV: {mname} ...", end=" ", flush=True)
    df_m, preds_m = evaluate_rul_loro(runs_cal, mk, cap=RUL_CAP, late_thr=25.0)
    all_rows.append(df_m)
    all_preds[mname] = preds_m
    print(f"RMSE={df_m['rmse'].mean():.2f} ± {df_m['rmse'].std():.2f} | "
          f"MAE={df_m['mae'].mean():.2f} | PHM={df_m['phm_mean'].mean():.2f}")

rul_results = pd.concat(all_rows, ignore_index=True)


### Summary Table: Mean ± Std Across the 4 LORO Folds


In [ ]:
agg = rul_results.groupby("model")[["rmse", "mae", "phm_mean", "rmse_late", "mae_late"]].agg(["mean", "std"])
summary = pd.DataFrame(index=agg.index)
for col in ["rmse", "mae", "phm_mean", "rmse_late", "mae_late"]:
    summary[col] = [f"{agg.loc[m, (col, 'mean')]:.2f} ± {agg.loc[m, (col, 'std')]:.2f}" for m in agg.index]
summary["_sort"] = agg[("rmse", "mean")]
summary = summary.sort_values("_sort").drop(columns="_sort")
display(summary)

print("\nPer-bearing RMSE:")
display(rul_results.pivot(index="model", columns="run", values="rmse").round(2))


---
## 2. Predicted vs. True RUL Plots (Top Models + Baselines)


In [ ]:
SHOW_MODELS = ["hi_baseline", "ridge", "random_forest", "xgboost", "lightgbm", "lstm", "cnn_lstm", "exp_degradation"]
SHOW_MODELS = [m for m in SHOW_MODELS if m in all_preds]

fig, axes = plt.subplots(len(runs_cal), len(SHOW_MODELS), figsize=(3.0 * len(SHOW_MODELS), 3.0 * len(runs_cal)), sharey=True)
for r, rname in enumerate(runs_cal):
    life = z_runs[rname]["life"].to_numpy()
    for c, mname in enumerate(SHOW_MODELS):
        ax = axes[r, c]
        y_true, y_pred = all_preds[mname][rname]
        ax.plot(life, y_true, "k-", lw=1.2, label="True RUL")
        ax.plot(life, y_pred, color="tab:blue", lw=0.7, alpha=0.85, label="Predicted")
        ax.set_ylim(-2, RUL_CAP + 4)
        ax.tick_params(labelsize=7)
        if r == 0:
            ax.set_title(mname, fontsize=9, fontweight="bold")
        if c == 0:
            ax.set_ylabel(f"{rname}\nRUL (h)", fontsize=8)
        if r == len(runs_cal) - 1:
            ax.set_xlabel("Life fraction", fontsize=8)
        if r == 0 and c == 0:
            ax.legend(fontsize=7, loc="lower left")

fig.suptitle("Predicted vs. True RUL (Leave-One-Run-Out CV, Cap = 50 h)", fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(str(results_dir / "rul_pred_vs_true.png"), dpi=130, bbox_inches="tight")
plt.show()


---
## 3. Error as a Function of Time-to-Failure (Late-Life Accuracy)


In [ ]:
bins = [0, 10, 20, 30, 40, 50.1]
bin_labels = ["0-10h", "10-20h", "20-30h", "30-40h", "40-50h"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: RMSE by RUL bin for representative models
for mname in ["ridge", "random_forest", "xgboost", "lightgbm", "lstm", "exp_degradation"]:
    if mname not in all_preds:
        continue
    ys_t = np.concatenate([all_preds[mname][r][0] for r in runs_cal])
    ys_p = np.concatenate([all_preds[mname][r][1] for r in runs_cal])
    df_tmp = pd.DataFrame({"y": ys_t, "err2": (ys_p - ys_t) ** 2})
    df_tmp["bin"] = pd.cut(df_tmp["y"], bins=bins, labels=bin_labels, include_lowest=True)
    bin_rmse = np.sqrt(df_tmp.groupby("bin", observed=False)["err2"].mean())
    ax1.plot(bin_labels, bin_rmse.values, marker="o", label=mname)

ax1.set_xlabel("True RUL bin (hours before failure)")
ax1.set_ylabel("RMSE (hours)")
ax1.set_title("Error vs. Time-to-Failure")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# Right: Overall RMSE vs Late-life RMSE (RUL <= 25h) bar comparison
mean_df = rul_results.groupby("model")[["rmse", "rmse_late"]].mean().sort_values("rmse")
mean_df.plot.barh(ax=ax2, color=["#2196F3", "#FF5722"], edgecolor="k", linewidth=0.5)
ax2.set_xlabel("Mean RMSE across 4 folds (hours)")
ax2.set_title("Overall RMSE vs. Late-Life RMSE (RUL ≤ 25 h)")
ax2.legend(["Overall RMSE", "Late-life RMSE (≤25h)"])

fig.tight_layout()
fig.savefig(str(results_dir / "rul_error_vs_ttf.png"), dpi=130, bbox_inches="tight")
plt.show()


---
## 4. Ablations: Prediction Smoothing & Operating-Hours Labels


In [ ]:
# Ablation A: Causal rolling-median smoothing (win=15) on predictions
smooth_rows = []
for mname in RUL_MAKERS:
    for rname in runs_cal:
        y_t, y_p = all_preds[mname][rname]
        y_ps = smooth_predictions(y_p, win=15)
        smooth_rows.append({
            "model": mname, "run": rname,
            "rmse_raw": rmse(y_t, y_p), "rmse_smooth": rmse(y_t, y_ps),
            "phm_raw": phm_score(y_t, y_p), "phm_smooth": phm_score(y_t, y_ps),
        })
ab_smooth = pd.DataFrame(smooth_rows).groupby("model").mean(numeric_only=True).round(3)
ab_smooth["rmse_delta"] = (ab_smooth["rmse_smooth"] - ab_smooth["rmse_raw"]).round(3)
print("=== Smoothing Ablation (win=15 causal median) ===")
display(ab_smooth.sort_values("rmse_smooth"))

# Ablation B: Calendar hours vs Operating hours (on top fast models)
op_rows = []
for mname in ["ridge", "random_forest", "xgboost", "lightgbm", "lstm"]:
    df_op, _ = evaluate_rul_loro(runs_op, RUL_MAKERS[mname], cap=RUL_CAP, late_thr=25.0)
    op_rows.append(df_op)
ab_op = pd.concat(op_rows).groupby("model")[["rmse", "mae", "phm_mean"]].mean().round(2)
ab_cal = rul_results[rul_results["model"].isin(ab_op.index)].groupby("model")[["rmse", "mae", "phm_mean"]].mean().round(2)
comp = ab_cal.join(ab_op, lsuffix="_cal", rsuffix="_op")
print("\n=== Calendar vs. Operating Hours Ablation ===")
display(comp)
